# Student GPA Prediction — Machine Learning Modeling

---

## Modeling Objective

This notebook builds a regression model to predict `Post_Semester_GPA` using academic background, GenAI usage behavior, study habits, and psychological factors.

Two experiments isolate the effect of prior academic performance:
1. **Experiment A (With `Pre_Semester_GPA`)**: Represents a standard scenario. Prior GPA is typically a dominant predictor.
2. **Experiment B (Without `Pre_Semester_GPA`)**: Tests if GenAI usage and behavioral features hold independent predictive value.

Models are evaluated using MAE, RMSE, and R² because the target is continuous. MAE provides a direct estimate of the average GPA-point prediction error.

> **Note:** Exploratory Data Analysis is in `01_eda.ipynb`. This notebook focuses entirely on model training, error analysis, and deployment preparation.

## 1. Environment Setup

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from copy import deepcopy

from sklearn.model_selection import train_test_split, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance

import joblib
import sklearn

# ── Configuration ────────────────────────────────────────────────────────
# Paths – works in any local IDE when notebook lives in notebooks/
PROJECT_PATH = Path.cwd().parent
DATA_PATH = PROJECT_PATH / "data" / "ai_student_impact_dataset.csv"
MODEL_DIR = PROJECT_PATH / "models"
RESULTS_DIR = PROJECT_PATH / "docs"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Plot style
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 12

# Display
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.4f}".format)

print(f"Project root    : {PROJECT_PATH}")
print(f"Dataset path    : {DATA_PATH}")
print(f"Model directory : {MODEL_DIR}")
print(f"scikit-learn    : {sklearn.__version__}")
print(f"Dataset exists  : {DATA_PATH.exists()}")

---
## 2. Data Loading & Validation

The dataset is loaded and validated to ensure all required features are present before modeling begins. The data is pre-cleaned with no missing values or duplicates, allowing us to proceed directly to feature engineering.

In [ ]:
df = pd.read_csv(DATA_PATH)
print(f"Dataset shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
df.head()

In [ ]:
# Validate required columns
required_columns = [
    "Student_ID", "Major_Category", "Year_of_Study", "Pre_Semester_GPA",
    "Weekly_GenAI_Hours", "Primary_Use_Case", "Prompt_Engineering_Skill",
    "Tool_Diversity", "Paid_Subscription", "Traditional_Study_Hours",
    "Perceived_AI_Dependency", "Institutional_Policy",
    "Anxiety_Level_During_Exams", "Post_Semester_GPA",
    "Skill_Retention_Score", "Burnout_Risk_Level"
]

missing_cols = [c for c in required_columns if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

print("✓ All 16 required columns present")
print(f"✓ Missing values : {df.isnull().sum().sum()}")
print(f"✓ Duplicate rows  : {df.duplicated().sum()}")

---
## 3. Feature & Target Definition

`Post_Semester_GPA` is the continuous target variable (1.0–4.0 scale). `Student_ID` is removed because it is a unique identifier with no predictive value. `Pre_Semester_GPA` is conditionally dropped based on the experiment being run.

In [ ]:
TARGET = "Post_Semester_GPA"
ID_COLUMN = "Student_ID"


def prepare_features(data, target_col=TARGET, use_pre_gpa=True):
    """
    Prepare feature matrix X and target vector y.

    Parameters
    ----------
    data : pd.DataFrame
    target_col : str – column to predict
    use_pre_gpa : bool – whether to include Pre_Semester_GPA

    Returns
    -------
    X : pd.DataFrame – features
    y : pd.Series – target
    """
    data = data.copy()
    drop_cols = [ID_COLUMN, target_col]
    if not use_pre_gpa:
        drop_cols.append("Pre_Semester_GPA")

    X = data.drop(columns=drop_cols)
    y = data[target_col]
    return X, y

---
## 4. Preprocessing Strategy

A scikit-learn `ColumnTransformer` handles feature scaling and encoding:
- **Numeric features:** Standardized to zero mean and unit variance. Median imputation is included to handle any unexpected nulls in production.
- **Categorical features:** One-hot encoded to allow linear and tree-based models to interpret them.

Preprocessing is wrapped in a `Pipeline` alongside the regressor. This ensures scaling and encoding are fitted solely on the training data, preventing data leakage during evaluation.

In [ ]:
def build_preprocessor(X):
    """Build preprocessing pipeline for numeric and categorical features."""
    numerical_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
    categorical_features = X.select_dtypes(include=["object", "bool"]).columns.tolist()

    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ])

    preprocessor = ColumnTransformer([
        ("num", numeric_pipeline, numerical_features),
        ("cat", categorical_pipeline, categorical_features)
    ])

    return preprocessor, numerical_features, categorical_features

---
## 5. Model Evaluation Utilities

In [ ]:
def evaluate_regression_model(pipeline, X_train, X_test, y_train, y_test):
    """Train a pipeline and return regression metrics on the test set."""
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)

    return {
        "MAE": mean_absolute_error(y_test, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_test, y_pred)),
        "R2": r2_score(y_test, y_pred),
        "Predictions": y_pred,
    }


def run_experiment(data, use_pre_gpa=True, experiment_name="Experiment"):
    """Run multiple regressors and compare performance."""
    X, y = prepare_features(data, use_pre_gpa=use_pre_gpa)
    preprocessor, numerical_features, categorical_features = build_preprocessor(X)

    print(f"\n{'='*60}")
    print(f"  {experiment_name}")
    print(f"{'='*60}")
    print(f"  Features before encoding : {X.shape[1]}")
    print(f"  Numerical features       : {numerical_features}")
    print(f"  Categorical features     : {categorical_features}")

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    regressors = {
        "Dummy Baseline": DummyRegressor(strategy="mean"),
        "Linear Regression": LinearRegression(),
        "Ridge Regression": Ridge(alpha=1.0),
        "Random Forest": RandomForestRegressor(
            n_estimators=200, random_state=42, n_jobs=-1
        ),
        "Gradient Boosting": GradientBoostingRegressor(
            n_estimators=200, learning_rate=0.05, max_depth=3, random_state=42
        ),
    }

    results = []
    trained_models = {}

    for name, reg in regressors.items():
        # Each model gets its OWN copy of the preprocessor (C4 fix)
        pipe = Pipeline([
            ("preprocessor", deepcopy(preprocessor)),
            ("regressor", reg),
        ])
        metrics = evaluate_regression_model(pipe, X_train, X_test, y_train, y_test)
        results.append({"Model": name, "MAE": metrics["MAE"],
                         "RMSE": metrics["RMSE"], "R2": metrics["R2"]})
        trained_models[name] = pipe

    results_df = pd.DataFrame(results).sort_values("RMSE").reset_index(drop=True)

    return {
        "results": results_df,
        "trained_models": trained_models,
        "X": X, "y": y,
        "X_train": X_train, "X_test": X_test,
        "y_train": y_train, "y_test": y_test,
        "numerical_features": numerical_features,
        "categorical_features": categorical_features,
    }

---
## 6. Experiment A — With `Pre_Semester_GPA`

The first experiment includes `Pre_Semester_GPA`. This establishes a baseline for how well the model can perform when comprehensive historical academic data is available.

In [ ]:
experiment_a = run_experiment(
    df, use_pre_gpa=True,
    experiment_name="Experiment A: With Pre_Semester_GPA"
)

experiment_a["results"]

---
## 7. Experiment B — Without `Pre_Semester_GPA`

The second experiment removes `Pre_Semester_GPA`. This isolates the remaining features—such as GenAI usage and study habits—to determine if they hold independent predictive power for academic outcomes.

In [ ]:
experiment_b = run_experiment(
    df, use_pre_gpa=False,
    experiment_name="Experiment B: Without Pre_Semester_GPA"
)

experiment_b["results"]

---
## 8. Experiment Comparison

In [ ]:
best_a = experiment_a["results"].iloc[0]
best_b = experiment_b["results"].iloc[0]

comparison_df = pd.DataFrame([
    {"Experiment": "A — With Pre_Semester_GPA",
     "Best Model": best_a["Model"], "MAE": best_a["MAE"],
     "RMSE": best_a["RMSE"], "R2": best_a["R2"]},
    {"Experiment": "B — Without Pre_Semester_GPA",
     "Best Model": best_b["Model"], "MAE": best_b["MAE"],
     "RMSE": best_b["RMSE"], "R2": best_b["R2"]},
])
comparison_df

In [ ]:
# Visual comparison
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
metrics = ["MAE", "RMSE", "R2"]
colors = ["#4c72b0", "#dd8452"]

for i, metric in enumerate(metrics):
    vals = [best_a[metric], best_b[metric]]
    bars = axes[i].bar(["Exp A\n(with Pre GPA)", "Exp B\n(without Pre GPA)"],
                       vals, color=colors, width=0.5, edgecolor="white")
    axes[i].set_title(metric, fontsize=13, fontweight="bold")
    axes[i].set_ylabel(metric)
    for bar, v in zip(bars, vals):
        axes[i].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                     f"{v:.3f}", ha="center", fontsize=11)

plt.suptitle("Best Model Performance — Experiment A vs B", fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

### Interpretation

Experiment A reached an R² of 0.914, meaning it explains most of the variation in post-semester GPA. Experiment B's performance dropped sharply to an R² of 0.070.

This contrast suggests that `Pre_Semester_GPA` is the dominant predictor, which is expected because prior academic performance is closely related to later outcomes. Without it, GenAI usage and behavioral features explain very little variance on their own.

> **Decision**: Experiment A's Gradient Boosting model is selected for deployment because it provides reliable predictions suitable for practical estimation.

---
## 9. Best Model Selection

In [ ]:
best_model_name = experiment_a["results"].iloc[0]["Model"]
best_pipeline = experiment_a["trained_models"][best_model_name]

print(f" Selected model: {best_model_name}")
print(f"  MAE  = {best_a['MAE']:.4f}")
print(f"  RMSE = {best_a['RMSE']:.4f}")
print(f"  R²   = {best_a['R2']:.4f}")

---
## 10. Cross-Validation

5-fold cross-validation is applied to the selected model to ensure the test-set performance is stable and not an artifact of a "lucky" train-test split.

In [ ]:
X_full = experiment_a["X"]
y_full = experiment_a["y"]

cv_results = cross_validate(
    deepcopy(best_pipeline), X_full, y_full, cv=5,
    scoring={
        "mae": "neg_mean_absolute_error",
        "rmse": "neg_root_mean_squared_error",
        "r2": "r2",
    },
    return_train_score=False,
)

cv_summary = pd.DataFrame({
    "Metric": ["MAE", "RMSE", "R²"],
    "Mean": [
        -cv_results["test_mae"].mean(),
        -cv_results["test_rmse"].mean(),
        cv_results["test_r2"].mean(),
    ],
    "Std": [
        cv_results["test_mae"].std(),
        cv_results["test_rmse"].std(),
        cv_results["test_r2"].std(),
    ],
})

print("── 5-Fold Cross-Validation Results ──")
print(cv_summary.to_string(index=False))

The cross-validation scores show low variance across folds, which suggests the model generalizes well to unseen data and is not severely overfitted.

---
## 11. Error Analysis

The model's prediction errors are analyzed below to check for systematic bias, heteroscedasticity, and variance across different GPA brackets.

In [ ]:
X_test = experiment_a["X_test"]
y_test = experiment_a["y_test"]
y_pred = best_pipeline.predict(X_test)
residuals = y_test - y_pred

In [ ]:
# 11a — Actual vs Predicted
fig, ax = plt.subplots(figsize=(8, 8))
sns.scatterplot(x=y_test, y=y_pred, alpha=0.25, s=12, color="steelblue", ax=ax)
ax.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()],
        "r--", alpha=0.7, label="Perfect Prediction (y = x)")
ax.set_title("Actual vs Predicted — Post-Semester GPA")
ax.set_xlabel("Actual GPA")
ax.set_ylabel("Predicted GPA")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# 11b — Residual Distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(residuals, bins=40, kde=True, color="steelblue", ax=axes[0])
axes[0].axvline(0, color="red", ls="--", alpha=0.6)
axes[0].set_title("Residual Distribution")
axes[0].set_xlabel("Residual (Actual − Predicted)")
axes[0].set_ylabel("Count")

# 11c — Residuals vs Predicted (check for heteroscedasticity)
sns.scatterplot(x=y_pred, y=residuals, alpha=0.2, s=10, color="coral", ax=axes[1])
axes[1].axhline(0, color="red", ls="--", alpha=0.6)
axes[1].set_title("Residuals vs Predicted Values")
axes[1].set_xlabel("Predicted GPA")
axes[1].set_ylabel("Residual")

plt.tight_layout()
plt.show()

print(f"Residual Mean: {residuals.mean():.4f}")
print(f"Residual Std : {residuals.std():.4f}")

In [ ]:
# 11d — Error by GPA Range
error_df = pd.DataFrame({"actual": y_test, "predicted": y_pred, "residual": residuals})
error_df["GPA_Range"] = pd.cut(error_df["actual"],
                                bins=[0, 2.5, 3.5, 4.01],
                                labels=["Low (<2.5)", "Medium (2.5–3.5)", "High (>3.5)"])

error_by_range = error_df.groupby("GPA_Range", observed=True).agg(
    count=("residual", "count"),
    mae=("residual", lambda x: np.abs(x).mean()),
    rmse=("residual", lambda x: np.sqrt((x**2).mean())),
    mean_residual=("residual", "mean"),
).reset_index()

print("── Error Breakdown by GPA Range ──")
print(error_by_range.to_string(index=False))

### Error Analysis Findings

The residuals are centered around zero and show a symmetric distribution, suggesting no systematic prediction bias. The Residuals vs Predicted plot lacks strong heteroscedastic patterns, meaning the model's error spread remains relatively constant across the GPA scale. 

The GPA range breakdown highlights that predictions may be slightly less reliable at the extremes (very high or very low GPAs) compared to the middle bracket.

---
## 12. Feature Importance

Feature influence is evaluated using two methods:
- **Tree-based (Gini) importance**: Extractable directly from the Gradient Boosting model.
- **Permutation importance**: A model-agnostic approach that measures the performance drop when a feature's values are randomly shuffled. This is generally more robust against high-cardinality bias.

In [ ]:
# Helper to extract feature names after one-hot encoding
def get_transformed_feature_names(pipeline, X):
    preprocessor = pipeline.named_steps["preprocessor"]
    num_feats = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
    cat_feats = X.select_dtypes(include=["object", "bool"]).columns.tolist()

    cat_pipe = preprocessor.named_transformers_["cat"]
    ohe_feats = cat_pipe.named_steps["onehot"].get_feature_names_out(cat_feats).tolist()

    return num_feats + ohe_feats

In [ ]:
# 12a — Tree-based feature importance
regressor = best_pipeline.named_steps["regressor"]
feature_names = get_transformed_feature_names(best_pipeline, X_full)

fi_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": regressor.feature_importances_
}).sort_values("Importance", ascending=False)

fig, ax = plt.subplots(figsize=(10, 7))
top = fi_df.head(15).sort_values("Importance")
sns.barplot(data=top, y="Feature", x="Importance", palette="viridis", ax=ax)
ax.set_title("Top 15 Features — Tree-Based (Gini) Importance")
ax.set_xlabel("Importance")
ax.set_ylabel("")
plt.tight_layout()
plt.show()

In [ ]:
# 12b — Permutation importance (more robust)
perm_result = permutation_importance(
    best_pipeline, X_test, y_test,
    n_repeats=10, random_state=42, n_jobs=-1, scoring="r2"
)

perm_df = pd.DataFrame({
    "Feature": X_test.columns,
    "Importance_Mean": perm_result.importances_mean,
    "Importance_Std": perm_result.importances_std,
}).sort_values("Importance_Mean", ascending=False)

fig, ax = plt.subplots(figsize=(10, 7))
top_perm = perm_df.head(15).sort_values("Importance_Mean")
ax.barh(top_perm["Feature"], top_perm["Importance_Mean"],
        xerr=top_perm["Importance_Std"], color="mediumseagreen", capsize=4)
ax.set_title("Top 15 Features — Permutation Importance (R² drop)")
ax.set_xlabel("Mean R² Decrease When Shuffled")
ax.set_ylabel("")
plt.tight_layout()
plt.show()

### Feature Importance Findings

`Pre_Semester_GPA` heavily dominates both importance measures. This aligns with the results from Experiment B, where removing this feature caused a severe performance drop. `Skill_Retention_Score` and `Traditional_Study_Hours` provide secondary predictive value.

GenAI-specific features (weekly hours, use cases, prompt skill) rank consistently low. This suggests they contribute very little to GPA prediction when accounting for traditional academic metrics.

> **Caution**: High feature importance does not imply causation. `Pre_Semester_GPA` is a strong predictor because prior performance correlates with future performance, not because previous grades directly "cause" later ones.

---
## 13. Model Card & Limitations

### Model Card

| Field | Details |
|---|---|
| **Model name** | Student GPA Prediction Model |
| **Model type** | Gradient Boosting Regressor (scikit-learn) |
| **Task** | Regression — predict `Post_Semester_GPA` |
| **Input features** | 14 features (7 numeric + 7 categorical) |
| **Output** | Predicted GPA (continuous, ~1.0–4.0 scale) |
| **Training data** | 50,000 student records (80% train, 20% test) |
| **Performance** | MAE ≈ 0.114, RMSE ≈ 0.144, R² ≈ 0.914 |
| **Primary use** | Estimating post-semester GPA for academic advising |

### Known Limitations

1. **Dominant predictor dependency:** ~91% of the model's accuracy comes from `Pre_Semester_GPA`. Without it, the model explains only ~7% of variance.
2. **Observational data only:** All relationships are correlational. The model cannot determine whether GenAI usage *causes* GPA changes.
3. **Self-reported features:** `Perceived_AI_Dependency` and `Anxiety_Level_During_Exams` are subjective measures with potential reporting bias.
4. **Potential temporal leakage:** `Skill_Retention_Score` and `Burnout_Risk_Level` may be measured concurrently with or after the target, which could inflate apparent predictive power.
5. **Dataset representativeness:** The dataset appears synthetic/simulated (zero missing values, perfectly balanced categories). Real-world generalizability is uncertain.
6. **No demographic features:** The model does not account for socioeconomic factors, which may confound results.

### Ethical Considerations

- Predictions should **not** be used as the sole basis for academic decisions affecting students.
- Model outputs are estimates, not guarantees. They should supplement — not replace — human judgment.
- Performance should be monitored for fairness across different majors and student populations.

---
## 14. Final Model Saving

Before exporting, the selected pipeline is refitted on the entire dataset (train and test). Since cross-validation confirmed stable generalization, this step maximizes the data available to the final deployed model.

> **Note**: The metrics reported in earlier sections were calculated on the held-out test set *prior* to this refit.

In [ ]:
# Create a fresh clone so the evaluated pipeline remains intact
final_model = deepcopy(best_pipeline)
final_model.fit(X_full, y_full)

model_path = MODEL_DIR / "student_gpa_prediction_model.pkl"
joblib.dump(final_model, model_path)

print(f"✓ Final model saved to: {model_path}")
print(f"  Model type : {type(final_model.named_steps['regressor']).__name__}")
print(f"  sklearn    : {sklearn.__version__}")

---
## 15. Deployment Readiness

### Expected Input Schema

The saved model expects a DataFrame with **exactly these 14 columns** (in any order):

| Feature | Type | Values / Range |
|---|---|---|
| `Major_Category` | string | `Arts`, `Business`, `Humanities`, `Medical`, `STEM` |
| `Year_of_Study` | string | `Freshman`, `Sophomore`, `Junior`, `Senior`, `Graduate` |
| `Pre_Semester_GPA` | float | 0.0 – 4.0 |
| `Weekly_GenAI_Hours` | float | 0.0 – 40.0 |
| `Primary_Use_Case` | string | `Copywriting/Drafting`, `Debugging/Troubleshooting`, `Direct_Answer_Generation`, `Ideation`, `Summarizing_Reading` |
| `Prompt_Engineering_Skill` | string | `Beginner`, `Intermediate`, `Advanced` |
| `Tool_Diversity` | int | 1 – 5 |
| `Paid_Subscription` | bool | `True`, `False` |
| `Traditional_Study_Hours` | float | 1.0 – 36.0 |
| `Perceived_AI_Dependency` | int | 1 – 10 |
| `Institutional_Policy` | string | `Actively_Encouraged`, `Allowed_With_Citation`, `Strict_Ban` |
| `Anxiety_Level_During_Exams` | int | 1 – 10 |
| `Skill_Retention_Score` | float | 10.0 – 100.0 |
| `Burnout_Risk_Level` | string | `Low`, `Medium`, `High` |

> **Critical:** The Streamlit app must send these **exact category values**. The OneHotEncoder uses `handle_unknown="ignore"`, which silently zeros out unknown categories instead of raising an error.

---
## 16. Test Prediction

The saved model is loaded and tested with a sample student profile. This validates that the pipeline correctly processes the expected categorical inputs and outputs a continuous GPA prediction.

In [ ]:
loaded_model = joblib.load(model_path)

# Sample student using EXACT category values from the training dataset
sample_student = pd.DataFrame({
    "Major_Category": ["STEM"],
    "Year_of_Study": ["Junior"],
    "Pre_Semester_GPA": [3.40],
    "Weekly_GenAI_Hours": [6.0],
    "Primary_Use_Case": ["Debugging/Troubleshooting"],
    "Prompt_Engineering_Skill": ["Intermediate"],
    "Tool_Diversity": [3],
    "Paid_Subscription": [True],
    "Traditional_Study_Hours": [12.0],
    "Perceived_AI_Dependency": [4],
    "Institutional_Policy": ["Allowed_With_Citation"],
    "Anxiety_Level_During_Exams": [3],
    "Skill_Retention_Score": [78.0],
    "Burnout_Risk_Level": ["Medium"],
})

predicted_gpa = loaded_model.predict(sample_student)[0]
print(f"✓ Predicted Post-Semester GPA: {predicted_gpa:.2f}")

## 17. Save Experiment Results

In [ ]:
experiment_a["results"].to_csv(RESULTS_DIR / "model_results_with_pre_gpa.csv", index=False)
experiment_b["results"].to_csv(RESULTS_DIR / "model_results_without_pre_gpa.csv", index=False)
comparison_df.to_csv(RESULTS_DIR / "experiment_comparison.csv", index=False)

print(f"✓ Results saved to {RESULTS_DIR}")

---
## 18. Conclusion

### Summary

A Gradient Boosting Regressor was selected to predict `Post_Semester_GPA`. The model reached an R² of 0.914 when prior GPA was included, but performance dropped to an R² of 0.070 when it was removed. Cross-validation showed stable results across folds, and the final pipeline was exported as a `.pkl` file for deployment.

### Key Takeaway

`Pre_Semester_GPA` appears to be the primary driver of the model's predictions. GenAI usage, study habits, and psychological factors provide only marginal predictive value for this specific regression task. 

While these behavioral features might influence learning quality and student well-being, the data suggests they cannot reliably forecast GPA independently of historical academic performance.

### Next Steps

- Integrate the serialized model into the Streamlit interface (`app/streamlit_app.py`).
- Monitor live predictions to ensure the model responds reasonably to real-world user inputs.
- Consider exploring causal inference techniques to better understand the relationship between GenAI usage and academic stress, rather than treating it purely as a predictive modeling problem.